<a href="https://colab.research.google.com/github/veer94/sample/blob/main/Voters-Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [34]:
!pip install pymupdf

import fitz # PyMuPDF
import os

pdf_path = 'check.pdf'
output_dir = 'temp_pages'

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

doc = fitz.open(pdf_path)
for page_num in range(len(doc)):
    page = doc.load_page(page_num)
    output_pdf_path = os.path.join(output_dir, f'page_{page_num + 1}.pdf')
    page_doc = fitz.open()
    page_doc.insert_pdf(doc, from_page=page_num, to_page=page_num)
    page_doc.save(output_pdf_path)
    page_doc.close()

doc.close()
print(f"PDF split into {len(doc)} pages.")

ValueError: document closed

In [30]:
!pip install google-generativeai

In [31]:
import os
os.environ

environ{'SHELL': '/bin/bash',
        'NV_LIBCUBLAS_VERSION': '12.5.3.2-1',
        'NVIDIA_VISIBLE_DEVICES': 'all',
        'COLAB_JUPYTER_TRANSPORT': 'ipc',
        'NV_NVML_DEV_VERSION': '12.5.82-1',
        'NV_CUDNN_PACKAGE_NAME': 'libcudnn9-cuda-12',
        'CGROUP_MEMORY_EVENTS': '/sys/fs/cgroup/memory.events /var/colab/cgroup/jupyter-children/memory.events',
        'NV_LIBNCCL_DEV_PACKAGE': 'libnccl-dev=2.22.3-1+cuda12.5',
        'NV_LIBNCCL_DEV_PACKAGE_VERSION': '2.22.3-1',
        'VM_GCE_METADATA_HOST': '169.254.169.253',
        'MODEL_PROXY_HOST': 'https://mp.kaggle.net',
        'HOSTNAME': '64fbcdb7825f',
        'LANGUAGE': 'en_US',
        'TBE_RUNTIME_ADDR': '172.28.0.1:8011',
        'COLAB_TPU_1VM': '',
        'GCE_METADATA_TIMEOUT': '3',
        'NVIDIA_REQUIRE_CUDA': 'cuda>=12.5 brand=unknown,driver>=470,driver<471 brand=grid,driver>=470,driver<471 brand=tesla,driver>=470,driver<471 brand=nvidia,driver>=470,driver<471 brand=quadro,driver>=470,driver<471 brand=

In [32]:
import google.generativeai as genai
import json
import re
import os
import pathlib
import mimetypes
import pandas as pd
from google.colab import userdata

# This list will hold all the extracted data from every page.
all_extracted_data = []

# Your API key and model setup from previous steps
genai.set_api_key(userdata.get('Google_API_Key')) # Set API key first
model = genai.GenerativeModel('gemini-1.5-flash') # Initialize the model

# Define the folder where you saved the single-page PDFs.
pages_dir = 'temp_pages'

AttributeError: module 'google.generativeai' has no attribute 'set_api_key'

For Reading the PDF File

In [ ]:
# The refined prompt for single-page extraction.
prompt = """
From this PDF page, please extract the following 6 values for each person listed:
1. Serial Number (क्रमांक)
2. Code (e.g., SMW2807659, MNQ......., etc)
3. Nirvachak ka naam
4. Husband's name / Father's name (पिता/पति का नाम)
5. Age (उम्र)
6. Gender (लिंग)

Provide the extracted data in a single JSON array where each object represents one person and contains the 6 specified keys.
"""

# Iterate over each file in the directory.
for filename in sorted(os.listdir(pages_dir)):
    if filename.endswith('.pdf'):
        file_path = pathlib.Path(os.path.join(pages_dir, filename))
        print(f"Processing {filename}...")

        # Read the file's binary content.
        file_data = file_path.read_bytes()
        mime_type, _ = mimetypes.guess_type(file_path)
        if mime_type is None:
            mime_type = 'application/pdf'

        # Prepare the content for the API call.
        contents = [
            prompt,
            {
                'mime_type': mime_type,
                'data': file_data
            }
        ]

        try:
            # Call the Gemini API.
            response = model.generate_content(contents)

            # Clean and parse the JSON string.
            json_data_string = response.text.strip().replace('```json', '').replace('```', '')

            # This is the crucial line to fix the "Husband's" key issue.
            json_data_string = json_data_string.replace("'s", "'s")

            # Handle potential JSONDecodeError.
            page_data = json.loads(json_data_string)

            # Append the extracted data from this page to the master list.
            all_extracted_data.extend(page_data)
            print(f"Extracted {len(page_data)} records from {filename}.")

        except json.JSONDecodeError as e:
            print(f"Error parsing JSON from {filename}: {e}")
            print(f"Raw response: {response.text[:200]}...")
        except Exception as e:
            print(f"An unexpected error occurred while processing {filename}: {e}")

print("\n--- Processing Complete ---")
print(f"Total number of records extracted from all pages: {len(all_extracted_data)}")

Creating the API Content

In [ ]:
# Create a Pandas DataFrame from the combined data.
df = pd.DataFrame(all_extracted_data)

# Specify the output filename.
output_file = 'voter_data-1.xlsx'

# Save the DataFrame to an Excel file.
df.to_excel(output_file, index=False)

print(f"Data successfully saved to {output_file}.")